# ZAIC 9B — upload đã kiểm hash, mặc định preflight

Gắn ZAIC_CODE_ONLY.zip rồi chạy kiểm tra nguồn + 99 tests CPU. Base và SFT là hai input riêng, Private. Chỉ sau preflight mới xem xét GPU smoke. Không chạy notebook Colab lịch sử.


In [ ]:
CONFIG = {'ACTION': 'preflight', 'BASE_INPUT': '', 'SFT_INPUT': '', 'GPU_RUN_CONFIRMED': False, 'COMPETITION_CLOUD_USE_ALLOWED': True, 'CLOUD_PERMISSION_EVIDENCE': 'User confirmation recorded in local CLOUD_PERMISSION_RECORD.json; independently unverified', 'TRAIN_FRESH_ADAPTER': False, 'BASE_FILE_HASHES': {'README.md': 'c5f5a8c2dddab69cfbf05279235aa5fddb137939a06539c4c7637aa900fef6d0', 'LICENSE': 'bbedc3fda3305820b977265f01b8619d87570a6739de3a5582c3464840f1e57a', 'config.json': 'd0883072e01861ed0b2d47be3c16c36a8e81c224c7ffaa310c6558fb3f932b05', 'chat_template.jinja': 'a4aee8afcf2e0711942cf848899be66016f8d14a889ff9ede07bca099c28f715', 'merges.txt': 'a9d356d7bdf1ef4949e3e748e95b8e10ad9d4e2e838eddc38a0a7b6b94d1db8d', 'model.safetensors-00002-of-00004.safetensors': '31c7d7e2dd5d207840b31cc59083c8f4c4718959149e0358c0364052bb9a0330', 'model.safetensors-00001-of-00004.safetensors': 'db6f444b43d318c92f360a13a25561a6a65b10c0631b8ed305a426dbaa6c380e', 'model.safetensors-00004-of-00004.safetensors': 'b62b0c4cd7e44edee103ee8f4fe225f246d5e768e07bfd5f25b63a8aa1fdd0c6', 'model.safetensors.index.json': '26d3539b516be613f39563617cb9d33b3f83d401298125be392c80cefb8f7fe5', 'preprocessor_config.json': '27225450ac9c6529872ee1924fcb0962ff5634834f817040f444118116f4e516', 'tokenizer.json': '5f9e4d4901a92b997e463c1f46055088b6cca5ca61a6522d1b9f64c4bb81cb42', 'tokenizer_config.json': '316230d6a809701f4db5ea8f8fc862bc3a6f3229c937c174e674ff3ca0a64ac8', 'video_preprocessor_config.json': '7768af27c1fafa9cc9011c1dc20067e03f8915e03b63504550e11d5066986d13', 'vocab.json': 'ce99b4cb2983d118806ce0a8b777a35b093e2000a503ebde25853284c9dfa003', 'model.safetensors-00003-of-00004.safetensors': '7ec36ba3a4176a44c3c0876ad80c56a2f70c84bf008d82e9501df642f17dadec'}}


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, stat, sys, zipfile
EXPECTED_CODE_FILES = {'frozen_v21/app/abstention_policy.py': '4e9a8d56c85ac694ca830f1997ef040780309a3ee4a340e82b2ed7af8bab9aa6', 'frozen_v21/app/build_data.py': '5d9cdccb31911aac972db00134891b5dd6201fa6be30eaf5292146fcd5619816', 'frozen_v21/app/evaluate_saved.py': 'b9f9d2940db8cc5f556f4e0da32db41a47ac020d15c04a32a813396c9f3c4544', 'frozen_v21/app/infer.py': 'da7e0733cfa36da3190bd2c295dd26f319b468d8c61b9a4f0becb95454c51998', 'frozen_v21/app/legacy_engine.py': 'e79700a238552791b897b6f4512cbb55443401ab3a4fe6ea5c3708c6061d165d', 'frozen_v21/app/modeling.py': '5dfad538310cf25d5ab9ab900ff031e79fe649fd97f4cccf4c175a96a9e0d7ca', 'frozen_v21/app/offline_guard.py': 'e001aa904da64869b839b0eac9ac12b816037ba680480988af50114145956a8b', 'frozen_v21/app/planner.py': '3f34c6c74bd07796b4fcc6151e4edae1a227904dbb9adc29aaa6141ee61dbf5a', 'frozen_v21/app/scoring.py': 'df6dea2f122b0ff901e2fe0de299b3b6ef2dd29a69e1b12f9df0da8af51bd476', 'frozen_v21/app/smoke_model.py': 'f982fcd3cb2084a9304b4cf6abae58eeaf2b1e68d97049d3d342261307902463', 'frozen_v21/app/test_core.py': 'ac9490cff7c6639d7e9cf606617216cf76c6d51c98c076af7ec9c0ae7ef96586', 'frozen_v21/app/test_v2_regression.py': '9d9cef5cd46699201e4b3528e0815f7d769e92a75f2ec875d044d8fe354c5e39', 'frozen_v21/app/train.py': '23e5bbc3e5c1ebadaab6cbfc2f152dd52d5cb2dbd2642a02707fffc7c2368b65', 'frozen_v21/app/v2_contracts.py': 'ed5083fe93f24a387a82e37ce0b516d84a632bedfa42c8eceb8f2ae7eef9bb56', 'frozen_v21/app/workflow.py': 'aaa819632ecb6ae3d83e5c1a0ffbff18b81f1644d5ce0d7deb8198f0dc4020db', 'frozen_v21/model_lock.json': '6c7616ebe581cbf90a3b0d78f95631f4c907c1e65953f7c796375271447b2a21', 'frozen_v21/SOURCE_SHA256.json': 'a3587c1d6d441ef03fdcd79e0e34b3e014c8433d0368902314756ff270b17f01', 'frozen_v21/tools/audit_output.py': 'ba5c31a5db89e310d3f6a55a722f126776a1c5e7213c1e9d6a2c9d1de1616f90', 'frozen_v21/tools/fit_abstention.py': '9ae83592a43ea1775112b5ddb123ec39301509ba8cfb9ec93b0d30a294e8e985', 'frozen_v21/tools/infer_batched.py': '15a77486a272c916f0300f8314f7fb0331c7a1aea754f869d14a4abf5f171c35', 'frozen_v21/tools/inventory.py': 'eae123058d4a954c7e31460fe972735f73a3d04a9169c9c5df23cc28bb03050f', 'frozen_v21/tools/release_test.py': 'c43d16882e49f33c154f674a4c256e9a843407659ec5989044454b4ec1eb3c4a', 'frozen_v21/tools/replay_train.py': 'b45b36b7f98c19380e1a81a59019b3a96209eb7c08a5127018571034e428a169', 'frozen_v21/tools/run_repair_v2.py': 'bc5ef47750795a834427263fc0e9af0762fc917f8accd0404b7b72f0d4254f1e', 'frozen_v21/tools/run_repair_v21.py': 'a0eeec5e4e3951d04ebc8118cc3c9644363d4a9268146c4199c7495e2c106f82', 'frozen_v21/tools/test_audit_output.py': '0e75b3ac76986e513fa7981cefe7fd1dbae256ea7107bd6e56b42c3d0847192c', 'frozen_v21/tools/test_batch_io.py': 'd55abda01c3ecde28be2128597292313510a792475bb135d99109c5f35804814', 'frozen_v21/tools/test_policy.py': '94bc0795c912ab7d55426c402e0d3f557e1744f727883280c80b75de93e39e75', 'frozen_v21/tools/verify_ingestion.py': '9d0c5c050d1a0ab61a035d6057f0a7e4849e4a936e5d181daaa9971721efb114', 'gui/kaggle_runner.py': 'e2c469a9e97f9d7ca7ed205a12488d34e4ad0ee99b2d3f2f04aac206e5e28460', 'gui/local_check.pyw': '72497b8f989160bc6b20fff82ed460cd60d5ffd4510dcf87f2b7893ead28eb14', 'gui/run_sync.py': 'ee0a7608c1dfae4e3db22f8efb11d8fea087dd90275bfc07ddcc4cfc3cd1495b', 'gui/versioned_runner.py': '8ddbcf45608cb9fdf3f36c6fba08dd3068d7bf79104a51f7f47f52b627b8bc8d'}
archives=list(Path('/kaggle/input').rglob('ZAIC_CODE_ONLY.zip'))
if archives:
    assert len(archives)==1, 'Attach exactly one code dataset version'
    archive=archives[0]
    assert hashlib.sha256(archive.read_bytes()).hexdigest()=='b830ed5ecbf103423693f21a1d33b14d3ad41f0caf8c340a913407d9a253d8af', 'Code archive mismatch'
    source=Path('/kaggle/temp/zaic_source')/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    source.mkdir(parents=True,exist_ok=False)
    with zipfile.ZipFile(archive) as z:
        assert sum(i.file_size for i in z.infolist())<25*1024*1024
        for i in z.infolist():
            assert (source/i.filename).resolve().is_relative_to(source.resolve())
            assert not stat.S_ISLNK(i.external_attr>>16)
        z.extractall(source)
    source=source/'CODEX_PROJECT'
else:
    matches=list(Path('/kaggle/input').rglob('CODEX_PROJECT/gui/versioned_runner.py'))
    assert len(matches)==1, 'Attach the code ZIP or its extracted CODEX_PROJECT folder'
    source=matches[0].parents[1]
for name, expected_hash in EXPECTED_CODE_FILES.items():
    target=(source/name).resolve()
    assert target.is_relative_to(source.resolve())
    assert hashlib.sha256(target.read_bytes()).hexdigest()==expected_hash, 'Code file hash mismatch'
sys.path.insert(0,str(source/'gui'))
if CONFIG['ACTION']!='preflight':
    bases=list(Path('/kaggle/input').rglob('BASE_UPLOAD_MANIFEST.json'))
    assert len(bases)==1, 'Attach exactly one version of the base dataset'
    CONFIG['BASE_INPUT']=str(bases[0].parent/'base')
if CONFIG['ACTION']=='train':
    datasets=list(Path('/kaggle/input').rglob('SFT_UPLOAD_MANIFEST.json'))
    assert len(datasets)==1, 'Attach exactly one approved training dataset'
    CONFIG['SFT_INPUT']=str(datasets[0].parent/'data')
from versioned_runner import run
RESULT=run(CONFIG,source)


## Sau khi kiểm tra đạt
Smoke: ACTION=smoke, GPU_RUN_CONFIRMED=True. Train mới: ACTION=train và TRAIN_FRESH_ADAPTER=True sau khi xem xét cấu hình; không resume adapter cũ. Mặc định cả hai cổng này tắt.

Lưu version có outputs, tải ZIP run về máy, đối chiếu hash rồi mới LATEST_SYNCED. BEST_MODEL chọn bằng đánh giá TRAIN.
